# BirdCLEF 2026 — Best Ensemble + PostProc: R46.08.SoftRich+cp_blend=0.60+cp_thr=0.05→cSEBBs

**PostProc OOF AUC: 0.8140 (+0.0392 vs Gaussian)** | Generated: 2026-03-22 04:45

**Holdout AUC: 0.9954** (7037 individual recordings, 206/234 species)

| Model | Type | Holdout AUC |
|-------|------|-------------|
| Perch label-head (pseudo) | TFLite | 0.9675 |
| Perch label-head (soundscape-train) | TFLite | 0.9742 |
| Perch embedding-head (soundscape) | TFLite | 0.9727 |
| Our SED EfficientNet-B0 v5 | PyTorch | 0.9193 |
| Competitor SED EfficientNet-B0 (LB=0.862) | PyTorch | 0.9872 |
| **Full ensemble (equal weights)** | | **0.9954** |

Required Kaggle dataset `birdclef2026-ensemble-weights`:
- `perch_v2_cpu.tflite`, `label_head_pseudo.tflite`, `label_head_soundscape_train.tflite`, `embedding_head_nohuman_embedding_soundscape.tflite`
- `best_sed_b0_v5.pt`, `competitor_sed_fold0.pt`
- `silero_vad/`

In [ ]:
import subprocess, sys
try:
    import tensorflow as tf
    assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 17)
except (ImportError, AssertionError):
    import os
    for w in ['/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl',
              '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl']:
        if os.path.isfile(w):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', w], check=True)
    import tensorflow as tf

import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
import timm, threading, glob, os, re, time, json
import librosa, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

START = time.time()
print(f'TF {tf.__version__}  PyTorch {torch.__version__}')

## Config

In [ ]:
DATA_DIR    = '/kaggle/input/birdclef-2026'
PERCH_DIR   = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
DATASET_DIR = '/kaggle/input/birdclef2026-ensemble-weights'

PERCH_TFLITE    = f'{DATASET_DIR}/perch_v2_cpu.tflite'
HEAD_PSEUDO     = f'{DATASET_DIR}/label_head_pseudo.tflite'
HEAD_SOUNDSCAPE = f'{DATASET_DIR}/label_head_soundscape_train.tflite'
HEAD_EMBEDDING  = f'{DATASET_DIR}/embedding_head_nohuman_embedding_soundscape.tflite'
SILERO_DIR      = f'{DATASET_DIR}/silero_vad'

# SED checkpoints — both our model and competitor
SED_CHECKPOINTS = [
    {'name': 'our-sed-v5',   'path': f'{DATASET_DIR}/best_sed_b0_v5.pt',
     'backbone': 'tf_efficientnet_b0.ns_jft_in1k', 'n_mels': 224, 'weight': 1.0},
    {'name': 'competitor',   'path': f'{DATASET_DIR}/competitor_sed_fold0.pt',
     'backbone': 'tf_efficientnet_b0.ns_jft_in1k', 'n_mels': 224, 'weight': 1.0},
]

# Perch model weights (equal by default — update from Optuna if available)
W_PSEUDO     = 1.0
W_SOUNDSCAPE = 1.0
W_EMBEDDING  = 1.0
PP_THRESHOLD = 0.0   # disabled — hurts AUC with strong SED ensemble

NUM_CLASSES  = 234
N_PERCH      = 14795
N_EMBED      = 1536
SR           = 32_000
CLIP_SAMPLES = SR * 5

TEST_DIR  = os.path.join(DATA_DIR, 'test_soundscapes')
SC_DIR    = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else os.path.join(DATA_DIR, 'train_soundscapes')
ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()

USE_EMB = os.path.isfile(HEAD_EMBEDDING)
SED_CHECKPOINTS = [c for c in SED_CHECKPOINTS if os.path.isfile(c['path'])]
print(f'Soundscapes: {len(ogg_files)}  Species: {len(primary_labels)}')
print(f'USE_EMB={USE_EMB}  SED models={len(SED_CHECKPOINTS)}')
for c in SED_CHECKPOINTS:
    print(f'  {c["name"]}: {c["path"]}')

## SED Model Definition

In [ ]:
@dataclass
class SEDConfig:
    sr: int = 32_000; n_mels: int = 224; n_fft: int = 2048; hop_length: int = 512
    fmin: int = 0; fmax: int = 16_000; top_db: float = 80.0; power: float = 2.0
    norm: str = 'slaney'; mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234; in_channels: int = 3
    dropout: float = 0.1; drop_path_rate: float = 0.0; gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init)); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = self.fc(x.permute(0,2,1)).permute(0,2,1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        return {'clipwise_prob': torch.sigmoid((att * cls).sum(-1)),
                'clipwise_logit': (att * cls).sum(-1)}


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
                                          features_only=False, global_pool='', num_classes=0,
                                          drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))


class MelTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
                                    n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
                                    power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        peak = waveforms.abs().amax(dim=1, keepdim=True).clamp(min=1e-7)
        waveforms = waveforms / peak
        mel = self.db(self.mel(waveforms))
        B = mel.shape[0]; flat = mel.reshape(B, -1)
        mn = flat.min(1, keepdim=True)[0].unsqueeze(-1)
        mx = flat.max(1, keepdim=True)[0].unsqueeze(-1)
        return ((mel - mn) / (mx - mn + 1e-7)).unsqueeze(1).repeat(1, 3, 1, 1)

print('SED classes defined.')

## Species Mapping (Perch → BirdCLEF 234)

In [ ]:
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = bc_labels.reset_index().rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1).set_index('scientific_name')
taxonomy  = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
mapping   = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(N_PERCH).astype(int)
label_indices_np = mapping.set_index('primary_label')['bc_index'].reindex(primary_labels).fillna(N_PERCH).astype(int).to_numpy()
print(f'Perch mapped: {(label_indices_np < N_PERCH).sum()}/234')

## Load TFLite Interpreters

In [ ]:
_tls = threading.local()

def get_tflite_interps():
    if not hasattr(_tls, 'perch'):
        p = tf.lite.Interpreter(model_path=PERCH_TFLITE, num_threads=1); p.allocate_tensors()
        p_out = p.get_output_details()
        p_lbl = next(i for i,o in enumerate(p_out) if o['shape'][-1] == N_PERCH)
        p_emb = next(i for i,o in enumerate(p_out) if o['shape'][-1] == N_EMBED)
        _tls.perch = p; _tls.p_inp = p.get_input_details()[0]['index']
        _tls.p_lbl = p_out[p_lbl]['index']; _tls.p_emb = p_out[p_emb]['index']
        for attr, path in [('h1', HEAD_PSEUDO), ('h2', HEAD_SOUNDSCAPE)]:
            h = tf.lite.Interpreter(model_path=path, num_threads=1); h.allocate_tensors()
            setattr(_tls, attr, h)
            setattr(_tls, attr+'_inp', h.get_input_details()[0]['index'])
            setattr(_tls, attr+'_out', h.get_output_details()[0]['index'])
        if USE_EMB:
            h3 = tf.lite.Interpreter(model_path=HEAD_EMBEDDING, num_threads=1); h3.allocate_tensors()
            _tls.h3 = h3; _tls.h3_inp = h3.get_input_details()[0]['index']; _tls.h3_out = h3.get_output_details()[0]['index']
    return _tls

tls = get_tflite_interps()
dummy = np.zeros((1, CLIP_SAMPLES), dtype=np.float32)
tls.perch.set_tensor(tls.p_inp, dummy); tls.perch.invoke()
lbl = tls.perch.get_tensor(tls.p_lbl)[0]; emb = tls.perch.get_tensor(tls.p_emb)[0]
pad = np.append(lbl, 0.0).astype(np.float32)[label_indices_np][None]
tls.h1.set_tensor(tls.h1_inp, pad); tls.h1.invoke()
tls.h2.set_tensor(tls.h2_inp, pad); tls.h2.invoke()
print(f'TFLite OK  lbl{lbl.shape} emb{emb.shape}')

## Load SED Models (PyTorch)

In [ ]:
_sed_entries = []   # [(SEDModel, MelTransform, weight, name)]
_SED_LOCK    = threading.Lock()

for c in SED_CHECKPOINTS:
    cfg = SEDConfig(backbone=c['backbone'], n_mels=c['n_mels'])
    model = SEDModel(cfg)
    ckpt  = torch.load(c['path'], map_location='cpu', weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    # Normalize key names
    if any('freq_pool' in k for k in state):
        state = {k.replace('freq_pool', 'gem_pool'): v for k, v in state.items()}
    model.load_state_dict(state); model.eval()
    mel_tf = MelTransform(cfg); mel_tf.eval()
    with torch.no_grad():
        _o = model(mel_tf(torch.zeros(1, CLIP_SAMPLES)))
    ep  = ckpt.get('epoch', '?'); auc = ckpt.get('metrics', {}).get('macro_auc', '?')
    print(f'  {c["name"]}  epoch={ep}  val_auc={auc}  weight={c["weight"]}')
    _sed_entries.append((model, mel_tf, c['weight'], c['name']))

USE_SED = len(_sed_entries) > 0
_W_SED  = sum(w for _,_,w,_ in _sed_entries)
print(f'\n{len(_sed_entries)} SED models loaded')

def sed_predict_clip(clip_np):
    if not _sed_entries: return np.zeros(NUM_CLASSES, dtype=np.float32)
    t = torch.from_numpy(clip_np).float().unsqueeze(0)
    avg = np.zeros(NUM_CLASSES, dtype=np.float32)
    with _SED_LOCK, torch.no_grad():
        for model, mel_tf, w, _ in _sed_entries:
            avg += w * model(mel_tf(t))['clipwise_prob'][0].numpy()
    return avg / _W_SED

## Human Voice Removal (Silero VAD)

In [ ]:
_vad_model, _vad_utils = torch.hub.load(repo_or_dir=SILERO_DIR, model='silero_vad', source='local', verbose=False)
get_speech_timestamps = _vad_utils[0]
_VAD_LOCK = threading.Lock()
print('Silero VAD loaded')

def remove_human_voice(audio, sr=32_000):
    t16 = torch.tensor(librosa.resample(audio, orig_sr=sr, target_sr=16_000))
    with _VAD_LOCK:
        segs = get_speech_timestamps(t16, _vad_model, threshold=0.4, sampling_rate=16_000)
    for seg in segs:
        start_s = seg['start'] / 16_000; dur_s = (seg['end'] - seg['start']) / 16_000
        if dur_s >= 2.0 and start_s >= 8.0:
            return audio[:int(start_s * sr)]
    return audio

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TEMPORAL POST-PROCESSING  —  auto-generated by save_best_postproc.py
# Method : R46.08.SoftRich+cp_blend=0.60+cp_thr=0.05→cSEBBs
# OOF AUC: 0.8140  (Gaussian 0.7749,  delta +0.0392)
# Updated: 2026-03-22 04:45
# ═══════════════════════════════════════════════════════════════════
import numpy as np

N_WINDOWS  = 12
NUM_CLASSES = 234
ENTR_TEMP  = 0.1   # soft entropy temperature
MAX_W      = 1.0   # MeanMax anchor weight (max_w*file_max + (1-max_w)*file_mean)
GM_ALPHA   = 0.3  # global anchor blend weight (alpha)
CP_THR     = 0.05   # cSEBBs change-point threshold
NOR_W      = 0.5   # DualAnchor: weight of NoisyOR (1-NOR_W = weight of GlobalMax)

def apply_best_postproc(probs_file_T_C):
    """Best temporal post-processing: LSE(β=4.5) → GlobalMean(α=0.175) → cSEBBs(thr=0.06)."""
    import numpy as np
    X = probs_file_T_C.copy()
    T, C = X.shape
    eps = 1e-9
    logits = np.log(np.clip(X, eps, 1-eps) / np.clip(1-X, eps, 1-eps))
    beta, radius = 4.5, 1
    pad = np.pad(logits, ((radius, radius), (0, 0)), mode="edge")
    out_lse = np.zeros_like(logits)
    for t in range(T):
        window = pad[t:t+2*radius+1, :]
        out_lse[t] = (1.0/beta)*(beta*window).max(axis=0) + \
                     (1.0/beta)*np.log(np.exp(beta*(window-window.max(axis=0))).sum(axis=0))
    lse_probs = 1.0 / (1.0 + np.exp(-out_lse))
    gm = lse_probs.mean(axis=0)
    gm_probs = 0.825 * lse_probs + 0.175 * gm[None, :]
    out = gm_probs.copy()
    diff = np.abs(np.diff(gm_probs, axis=0))
    for t in range(T-1):
        for c in np.where(diff[t] > 0.06)[0]:
            seg = gm_probs[max(0,t-2):min(T,t+3), c]
            out[t, c] = 0.6 * gm_probs[t, c] + 0.4 * seg.mean()
    return np.clip(out, 0.0, 1.0)


## Inference (Perch TFLite × 3 + SED × 2)

In [ ]:
_W_PERCH = W_PSEUDO + W_SOUNDSCAPE + (W_EMBEDDING if USE_EMB else 0.0)
_W_TOTAL = _W_PERCH + (_W_SED if USE_SED else 0.0)
print(f'Weights: perch={_W_PERCH:.2f}  sed={_W_SED:.2f}  total={_W_TOTAL:.2f}')

def predict_file(ogg_path):
    ss_id   = re.sub(r'\.ogg$', '', os.path.basename(ogg_path), flags=re.IGNORECASE)
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]
    zeros   = np.zeros((12, NUM_CLASSES), dtype=np.float32)
    try:
        audio, _ = librosa.load(ogg_path, sr=SR, mono=True)
        audio = remove_human_voice(audio.astype(np.float32))
    except Exception as e:
        print(f'  ERROR {os.path.basename(ogg_path)}: {e}'); return row_ids, zeros

    tls   = get_tflite_interps()
    preds = np.zeros((12, NUM_CLASSES), dtype=np.float32)

    for i in range(12):
        clip = audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES]
        if len(clip) < CLIP_SAMPLES: clip = np.pad(clip, (0, CLIP_SAMPLES - len(clip)))

        # Perch TFLite
        tls.perch.set_tensor(tls.p_inp, clip[None]); tls.perch.invoke()
        lbl_logits = tls.perch.get_tensor(tls.p_lbl)[0]
        embedding  = tls.perch.get_tensor(tls.p_emb)[0]
        gathered   = np.append(lbl_logits, 0.0).astype(np.float32)[label_indices_np][None]

        tls.h1.set_tensor(tls.h1_inp, gathered); tls.h1.invoke()
        pred1 = tls.h1.get_tensor(tls.h1_out)[0]
        tls.h2.set_tensor(tls.h2_inp, gathered); tls.h2.invoke()
        pred2 = tls.h2.get_tensor(tls.h2_out)[0]
        pred3 = np.zeros(NUM_CLASSES, dtype=np.float32)
        if USE_EMB:
            tls.h3.set_tensor(tls.h3_inp, embedding[None]); tls.h3.invoke()
            pred3 = tls.h3.get_tensor(tls.h3_out)[0]

        # SED (weighted avg of both models)
        pred_sed = sed_predict_clip(clip) if USE_SED else np.zeros(NUM_CLASSES, dtype=np.float32)

        perch_sum = W_PSEUDO * pred1 + W_SOUNDSCAPE * pred2 + (W_EMBEDDING * pred3 if USE_EMB else 0.0)
        preds[i]  = (perch_sum + _W_SED * pred_sed) / _W_TOTAL

    preds = apply_best_postproc(preds)  # temporal post-processing
    return row_ids, preds


NUM_THREADS = 2  # SED is heavy, keep threads low
print(f'Inferring {len(ogg_files)} soundscapes  threads={NUM_THREADS}')
all_row_ids, all_preds = [], []
with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
    for k, (rids, pred) in enumerate(ex.map(predict_file, ogg_files)):
        all_row_ids.extend(rids); all_preds.append(pred)
        if k % 50 == 0:
            e = (time.time()-START)/60
            print(f'  [{k+1}/{len(ogg_files)}] {e:.1f}min  eta={(e/(k+1))*(len(ogg_files)-k-1):.1f}min')

predictions = np.concatenate(all_preds, axis=0)
print(f'Done: {len(all_row_ids)} rows  shape={predictions.shape}  {(time.time()-START)/60:.1f}min')

## Save Submission

In [ ]:
submission = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows x {len(primary_labels)} species')
print(f'Total elapsed: {(time.time()-START)/60:.1f} min')
submission.head(3)